# Xeno-canto Gathering

Build a reproducible download set using XC query tags.


In [1]:
import json
import sys
import time
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src" / "config.py").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "config.py").exists():
    raise FileNotFoundError(f"Couldn't find src/config.py from {Path.cwd()}")

sys.path.insert(0, str(repo_root))

from src.config import CONFIG
from src.dataset.utils.xeno_canto import write_raw_manifest_for_species, download_from_selected_manifest
from src.dataset.utils.selection import write_selected_manifests


In [2]:
DATA_DIR = Path(CONFIG.paths.data_dir)
RAW_DIR = DATA_DIR / CONFIG.paths.raw_dir
MANIFEST_DIR = DATA_DIR / CONFIG.paths.manifests_dir

for p in [RAW_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)
    
SPECIES_FILE = Path("../../species_list_small.json")
species_map = json.loads(SPECIES_FILE.read_text(encoding="utf-8"))
species_list = list(species_map.values())
species_list[:5]

[{'common_name': 'European Robin', 'sci_name': 'Erithacus rubecula'},
 {'common_name': 'Eurasian Blackbird', 'sci_name': 'Turdus merula'},
 {'common_name': 'Eurasian Wren', 'sci_name': 'Troglodytes troglodytes'},
 {'common_name': 'Eurasian Blue Tit', 'sci_name': 'Cyanistes caeruleus'},
 {'common_name': 'Great Tit', 'sci_name': 'Parus major'}]

In [3]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # or walk parents robustly

DATA_DIR = PROJECT_ROOT / CONFIG.paths.data_dir
RAW_DIR = DATA_DIR / CONFIG.paths.raw_dir
MANIFEST_DIR = DATA_DIR / CONFIG.paths.manifests_dir
print(MANIFEST_DIR, len(list(MANIFEST_DIR.glob("*_selected.csv"))))


/Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests 50


In [4]:
def build_query(sci_name: str) -> str:
    xc = CONFIG.xeno_canto
    tags = [
        f'sp:"{sci_name}"',
        "grp:birds",
        "area:europe",
        'q:">C"',                 
        f'len:">{xc.min_len}"',
        f'len:"<{xc.max_len}"',
    ]
    return " ".join(tags)


## Fetch metadata and store in manifest files


In [6]:
for entry in species_list:
    sci_name = entry["sci_name"]
    query = build_query(sci_name)

    out_csv = write_raw_manifest_for_species(
        sci_name=sci_name,
        query=query,
        manifest_dir=MANIFEST_DIR,
        per_page=500,
    )

    print(f"Wrote raw manifest: {out_csv}")


Wrote raw manifest: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests/erithacus_rubecula.csv
Wrote raw manifest: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests/turdus_merula.csv
Wrote raw manifest: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests/troglodytes_troglodytes.csv
Wrote raw manifest: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests/cyanistes_caeruleus.csv
Wrote raw manifest: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests/parus_major.csv


KeyboardInterrupt: 

In [ ]:
summary = write_selected_manifests(
    MANIFEST_DIR,
    target_per_species=250,             
    recordist_cap=30,
    prefer_countries=("Ireland", "United Kingdom"),
    selected_suffix="selected",
)

# Print a readable per-species summary
for fname, info in summary["per_file"].items():
    stats = info.get("stats", {})
    if not stats:
        print(fname, info)
        continue
    print(f"\n=== {fname} ===")
    print("selected:", stats["selected"], "/", stats["total_in"])
    print("IE/UK:", stats["preferred_selected"], f"({stats['preferred_selected_pct']:.1f}%)")
    print("month coverage:", stats["selected_month_coverage"])
    print("max by one recordist:", stats["max_selected_by_one_recordist"])
    print("top countries:", stats["selected_country_top"][:5])


=== accipiter_nisus.csv ===
selected: 127 / 127
IE/UK: 28 (22.0%)
month coverage: 10
max by one recordist: 10
top countries: [('United Kingdom', 24), ('France', 21), ('Sweden', 20), ('Poland', 17), ('Germany', 13)]

=== acrocephalus_schoenobaenus.csv ===
selected: 250 / 548
IE/UK: 124 (49.6%)
month coverage: 10
max by one recordist: 25
top countries: [('United Kingdom', 81), ('Ireland', 43), ('France', 32), ('Sweden', 23), ('Portugal', 14)]

=== aegithalos_caudatus.csv ===
selected: 250 / 712
IE/UK: 88 (35.2%)
month coverage: 12
max by one recordist: 27
top countries: [('United Kingdom', 71), ('Poland', 38), ('France', 26), ('Sweden', 21), ('Spain', 20)]

=== alcedo_atthis.csv ===
selected: 250 / 554
IE/UK: 83 (33.2%)
month coverage: 12
max by one recordist: 23
top countries: [('United Kingdom', 54), ('France', 41), ('Ireland', 29), ('Portugal', 29), ('Germany', 25)]

=== anthus_pratensis.csv ===
selected: 250 / 707
IE/UK: 137 (54.8%)
month coverage: 12
max by one recordist: 30
top co

In [ ]:
# downloaded_csv = download_from_selected_manifest(
#     sci_name="Erithacus rubecula",
#     manifest_dir=MANIFEST_DIR,
#     raw_audio_dir=RAW_DIR,
#     overwrite=False,   
# )
# print("Wrote:", downloaded_csv)


1 / 250
2 / 250
3 / 250
4 / 250
5 / 250
6 / 250
7 / 250
8 / 250
9 / 250
10 / 250
11 / 250
12 / 250
13 / 250
14 / 250
15 / 250
16 / 250
17 / 250
18 / 250
19 / 250
20 / 250
21 / 250
22 / 250
23 / 250
24 / 250
25 / 250
26 / 250
27 / 250
28 / 250
29 / 250
30 / 250
31 / 250
32 / 250
33 / 250
34 / 250
35 / 250
36 / 250
37 / 250
38 / 250
39 / 250
40 / 250
41 / 250
42 / 250
43 / 250
44 / 250
45 / 250
46 / 250
47 / 250
48 / 250
49 / 250
50 / 250
51 / 250
52 / 250
53 / 250
54 / 250
55 / 250
56 / 250
57 / 250
58 / 250
59 / 250
60 / 250
61 / 250
62 / 250
63 / 250
64 / 250
65 / 250
66 / 250
67 / 250
68 / 250
69 / 250
70 / 250
71 / 250
72 / 250
73 / 250
74 / 250
75 / 250
76 / 250
77 / 250
78 / 250
79 / 250
80 / 250
81 / 250
82 / 250
83 / 250
84 / 250
85 / 250
86 / 250
87 / 250
88 / 250
89 / 250
90 / 250
91 / 250
92 / 250
93 / 250
94 / 250
95 / 250
96 / 250
97 / 250
98 / 250
99 / 250
100 / 250
101 / 250
102 / 250
103 / 250
104 / 250
105 / 250
106 / 250
107 / 250
108 / 250
109 / 250
110 / 250
111 / 25

In [6]:
from pathlib import Path

SKIP_SCI = {
    "accipiter nisus",
    "acrocephalus schoenobaenus",
    "aegithalos caudatus",
    "alcedo atthis",
    "anthus pratensis",
    "apus apus",
    "buteo buteo",
    "carduelis carduelis",
    "chloris chloris",
    "cinclus cinclus",
    "coloeus monedula",
    "columba livia",
    "columba palumbus",
    "corvus corax",
    "corvus cornix",
    "corvus frugilegus",
    "cuculus canorus",
    "cyanistes caeruleus",
    "delichon urbicum",
    "dendrocopos major",
    "emberiza citrinella",
    "emberiza schoeniclus",
    "erithacus rubecula",
    "falco peregrinus",
    "falco tinnunculus",
    "fringilla coelebs",
    "garrulus glandarius",
    "hirundo rustica",
    "motacilla alba",
    "motacilla cinerea",
    "muscicapa striata",
    "oenanthe oenanthe",
    "parus major",
    "passer domesticus",
    "periparus ater",
    "phasianus colchicus",
    "phylloscopus trochilus",
    "pica pica",
    "prunella modularis",
    "regulus regulus",
    "saxicola rubicola",
    "spinus spinus",
    "streptopelia decaocto",
    "sturnus vulgaris",
    "sylvia atricapilla",
    "troglodytes troglodytes",
    "turdus merula",
    "turdus philomelos",
    "tyto alba",
}



def sci_name_from_selected_file(p: Path) -> str:
    name = p.name
    if not name.endswith("_selected.csv"):
        raise ValueError(f"Not a selected manifest: {name}")
    slug = name[:-len("_selected.csv")]
    return slug.replace("_", " ")

def downloaded_manifest_path(selected_path: Path) -> Path:
    return selected_path.with_name(selected_path.name.replace("_selected.csv", "_downloaded.csv"))

selected_files = sorted(Path(MANIFEST_DIR).glob("*_selected.csv"))
print("Found selected manifests:", len(selected_files))

downloaded = []
for f in selected_files:
    sci = sci_name_from_selected_file(f)
    downloaded_csv = downloaded_manifest_path(f)

    if sci in SKIP_SCI:
        if downloaded_csv.exists():
            downloaded.append(downloaded_csv)
        print(f"Skipping {sci}")
        continue

    if downloaded_csv.exists():
        print(f"Already downloaded: {sci}")
        downloaded.append(downloaded_csv)
        continue

    print()
    print(f"=== Downloading {sci} ===")
    try:
        out = download_from_selected_manifest(
            sci_name=sci,
            manifest_dir=MANIFEST_DIR,
            raw_audio_dir=RAW_DIR,
            overwrite=False,
        )
    except Exception as exc:
        print(f"Failed {sci}: {exc}")
        continue
    downloaded.append(out)

print()
print("Done. Downloaded manifests written:")
for p in downloaded:
    print(" -", p)


Found selected manifests: 50
Skipping accipiter nisus
Skipping acrocephalus schoenobaenus
Skipping aegithalos caudatus
Skipping alcedo atthis
Skipping anthus pratensis
Skipping apus apus
Skipping buteo buteo
Skipping carduelis carduelis
Skipping chloris chloris
Skipping cinclus cinclus
Skipping coloeus monedula
Skipping columba livia
Skipping columba palumbus
Skipping corvus corax
Skipping corvus cornix
Skipping corvus frugilegus
Skipping cuculus canorus
Skipping cyanistes caeruleus
Skipping delichon urbicum
Skipping dendrocopos major
Skipping emberiza citrinella
Skipping emberiza schoeniclus
Skipping erithacus rubecula
Skipping falco peregrinus
Skipping falco tinnunculus
Skipping fringilla coelebs
Skipping garrulus glandarius
Skipping hirundo rustica
Skipping motacilla alba
Skipping motacilla cinerea
Skipping muscicapa striata
Skipping oenanthe oenanthe
Skipping parus major
Skipping passer domesticus
Skipping periparus ater
Skipping phasianus colchicus

=== Downloading phylloscopus co